In [7]:
import os
import json
import glob
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from pathlib import Path
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    roc_auc_score, precision_recall_curve,
    average_precision_score, confusion_matrix, ConfusionMatrixDisplay
)
import tensorflow as tf
from tensorflow.keras import layers, Model, callbacks
import joblib
 
warnings.filterwarnings("ignore")
tf.get_logger().setLevel("ERROR")

In [6]:
%pip install scikit-learn
%pip install tensorflow


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.2.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


  Obtaining dependency information for tensorflow from https://files.pythonhosted.org/packages/8f/a2/6d7e6a738e302530586d484895de2cf3fc158ad9c73b4504a670b2956dd9/tensorflow-2.21.0-cp311-cp311-win_amd64.whl.metadata
  Obtaining dependency information for absl-py>=1.0.0 from https://files.pythonhosted.org/packages/18/a6/907a406bb7d359e6a63f99c313846d9eec4f7e6f7437809e03aa00fa3074/absl_py-2.4.0-py3-none-any.whl.metadata
  Obtaining dependency information for astunparse>=1.6.0 from https://files.pythonhosted.org/packages/2b/03/13dde6512ad7b4557eb792fbcf0c653af6076b81e5941d36ec61f7ce6028/astunparse-1.6.3-py2.py3-none-any.whl.metadata
  Obtaining dependency information for flatbuffers>=25.9.23 from https://files.pythonhosted.org/packages/e8/2d/d2a548598be01649e2d46231d151a6c56d10b964d94043a335ae56ea2d92/flatbuffers-25.12.19-py2.py3-none-any.whl.metadata
  Obtaining dependency information for gast!=0.5.0,!=0.5.1,!=0.5.2,>=0.2.1 from https://files.pythonhosted.org/packages/1d/33/f1c6a276de27b7

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 23.2.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [8]:
# ─────────────────────────────────────────────
# CONFIG
# ─────────────────────────────────────────────
CFG = {
    "data_dir"        : "data",
    "output_dir"      : "outputs/m4",
    "sequence_len"    : 30,          # sliding window size (seconds)
    "test_ratio"      : 0.2,
    "ae_epochs"       : 80,
    "ae_batch"        : 64,
    "ae_latent_dim"   : 8,
    "ae_lr"           : 1e-3,
    "if_n_estimators" : 200,
    "if_contamination": 0.05,        # expected anomaly fraction in training
    "anomaly_pct_threshold": 95,     # reconstruction error percentile → threshold
    "random_seed"     : 42,
    "iqr_multiplier"  : 3.0,         # IQR clipping multiplier
    "rolling_window"  : 10,          # rolling stats window (seconds)
    "corr_threshold"  : 0.95,        # correlation filter threshold
    "perm_n_repeats"  : 5,           # permutation importance repeats
}
 
FEATURES = [
    "ENGINE_RPM", "VEHICLE_SPEED", "THROTTLE", "ENGINE_LOAD",
    "COOLANT_TEMP", "INTAKE_AIR_TEMP", "INTAKE_MANIFOLD_PRESSURE",
    "SHORT_TERM_FUEL_TRIM", "LONG_TERM_FUEL_TRIM",
    "ACCELERATOR_POS_D", "ACCELERATOR_POS_E", "ABSOLUTE_THROTTLE_B",
    "TIMING_ADVANCE", "CONTROL_MODULE_VOLTAGE", "FUEL_TANK_LEVEL",
    "FUEL_AIR_EQUIV_RATIO", "RELATIVE_THROTTLE_POS",
]
 
OUT = Path(CFG["output_dir"])
OUT.mkdir(parents=True, exist_ok=True)
np.random.seed(CFG["random_seed"])
tf.random.set_seed(CFG["random_seed"])

In [9]:
# ─────────────────────────────────────────────
# 1. DATA LOADING
# ─────────────────────────────────────────────
def load_csv_folder(folder: str, label: int) -> pd.DataFrame:
    files = glob.glob(os.path.join(folder, "*.csv"))
    if not files:
        raise FileNotFoundError(f"No CSV files found in {folder}")
    frames = []
    for f in files:
        df = pd.read_csv(f)
        df.columns = df.columns.str.strip().str.upper().str.replace(" ", "_")
        # Strip the "_()" suffix left by OBD-II headers like "ENGINE_RPM ()"
        df.columns = df.columns.str.replace(r"_\(\)$", "", regex=True)
        # Map dataset-specific names to the canonical FEATURES names
        col_map = {
            "COOLANT_TEMPERATURE": "COOLANT_TEMP",
            "SHORT_TERM_FUEL_TRIM_BANK_1": "SHORT_TERM_FUEL_TRIM",
            "LONG_TERM_FUEL_TRIM_BANK_1": "LONG_TERM_FUEL_TRIM",
            "PEDAL_D": "ACCELERATOR_POS_D",
            "PEDAL_E": "ACCELERATOR_POS_E",
            "FUEL_TANK": "FUEL_TANK_LEVEL",
            "FUEL_AIR_COMMANDED_EQUIV_RATIO": "FUEL_AIR_EQUIV_RATIO",
            "RELATIVE_THROTTLE_POSITION": "RELATIVE_THROTTLE_POS",
        }
        df.rename(columns=col_map, inplace=True)
        df["label"] = label          # 0 = normal, 1 = anomaly
        df["source_file"] = os.path.basename(f)
        frames.append(df)
    return pd.concat(frames, ignore_index=True)
 
 
def load_dataset() -> tuple[pd.DataFrame, pd.DataFrame]:
    drive = load_csv_folder(os.path.join(CFG["data_dir"], "drive"), label=0)
    idle  = load_csv_folder(os.path.join(CFG["data_dir"], "idle"),  label=0)
    normal = pd.concat([drive, idle], ignore_index=True)
    print(f"[data] normal rows: {len(normal):,}")
    return normal
 
 
def resolve_features(df: pd.DataFrame) -> list[str]:
    available = [f for f in FEATURES if f in df.columns]
    missing   = [f for f in FEATURES if f not in df.columns]
    if missing:
        print(f"[warn] missing features (will be skipped): {missing}")
    print(f"[data] using {len(available)} features: {available}")
    return available

In [ ]:
# ─────────────────────────────────────────────────────────────
# SECTION 2 — PREPROCESSING
# Steps (in order): resample → IQR clip → rolling stats → correlation filter
# ─────────────────────────────────────────────────────────────

def pp_resample(df: pd.DataFrame, features: list[str]) -> pd.DataFrame:
    """
    Step 1 — Resample to fixed Hz.
    Requires a TIMESTAMP column (seconds). If absent, data is assumed already
    at 1 Hz and only NaN cleaning is applied.
    """
    df = df.copy()
    if "TIMESTAMP" in df.columns:
        df["TIMESTAMP"] = pd.to_numeric(df["TIMESTAMP"], errors="coerce")
        df = df.dropna(subset=["TIMESTAMP"]).sort_values("TIMESTAMP")
        df = df.set_index(pd.to_timedelta(df["TIMESTAMP"], unit="s"))
        rule = f"{int(1 / CFG['target_hz'] * 1000)}ms"
        df = df[features + ["label"]].resample(rule).mean().interpolate("linear")
        df = df.reset_index(drop=True)
    else:
        df = df[features + ["label"]].reset_index(drop=True)

    before = len(df)
    df.replace([np.inf, -np.inf], np.nan, inplace=True)
    df[features] = df[features].ffill().bfill()
    df.dropna(inplace=True)
    print(f"[preprocess] 1/4 resample   → {len(df):,} rows  (dropped {before - len(df):,} NaN)")
    return df.reset_index(drop=True)


def pp_iqr_clip(df: pd.DataFrame, features: list[str]) -> pd.DataFrame:
    """
    Step 2 — IQR clipping.
    Clips extreme sensor spikes to [Q1 - k*IQR, Q3 + k*IQR].
    Preserves the distribution shape while removing hardware noise artifacts.
    """
    k = CFG["iqr_multiplier"]
    clip_stats, total_clipped = {}, 0
    for col in features:
        q1, q3 = df[col].quantile(0.25), df[col].quantile(0.75)
        iqr = q3 - q1
        lo, hi = q1 - k * iqr, q3 + k * iqr
        n_clipped = ((df[col] < lo) | (df[col] > hi)).sum()
        df[col] = df[col].clip(lo, hi)
        clip_stats[col] = {"lo": round(lo, 4), "hi": round(hi, 4), "clipped": int(n_clipped)}
        total_clipped += n_clipped

    print(f"[preprocess] 2/4 IQR clip   → {total_clipped:,} values clipped (k={k})")
    with open(OUT / "clip_stats.json", "w") as f:
        json.dump(clip_stats, f, indent=2)
    return df


def pp_rolling_statistics(df: pd.DataFrame, features: list[str]) -> tuple[pd.DataFrame, list[str]]:
    """
    Step 3 — Rolling statistics.
    Appends a rolling mean and rolling std column for every feature.
    Captures temporal dynamics (e.g. sustained high RPM vs a momentary spike)
    that raw instantaneous readings miss.
    """
    w, new_cols = CFG["rolling_window"], []
    for col in features:
        df[f"{col}_RMEAN"] = df[col].rolling(w, min_periods=1).mean()
        df[f"{col}_RSTD"]  = df[col].rolling(w, min_periods=1).std().fillna(0)
        new_cols.extend([f"{col}_RMEAN", f"{col}_RSTD"])
    extended = features + new_cols
    print(f"[preprocess] 3/4 rolling    → +{len(new_cols)} columns → {len(extended)} total (w={w}s)")
    return df, extended


def pp_correlation_filter(df: pd.DataFrame, features: list[str]) -> tuple[pd.DataFrame, list[str]]:
    """
    Step 4 — Drop highly correlated features (above CFG threshold).
    Keeps one representative from each correlated cluster.
    Reduces redundancy so both models learn from independent signals.
    """
    threshold = CFG["corr_threshold"]
    corr  = df[features].corr().abs()
    upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
    to_drop  = [col for col in upper.columns if any(upper[col] > threshold)]
    retained = [f for f in features if f not in to_drop]
    print(f"[preprocess] 4/4 corr filter → dropped {len(to_drop)}, retained {len(retained)}")
    if to_drop:
        print(f"             dropped: {to_drop}")

    fig, ax = plt.subplots(figsize=(max(10, len(retained)//2), max(8, len(retained)//2)))
    sns.heatmap(df[retained].corr(), ax=ax, cmap="coolwarm", center=0,
                annot=False, linewidths=0.4, cbar_kws={"shrink": 0.6})
    ax.set_title("Feature Correlation Matrix (post-filter)")
    fig.tight_layout()
    fig.savefig(OUT / "correlation_matrix.png", dpi=150)
    plt.close(fig)
    return df, retained


def preprocess(df: pd.DataFrame, raw_features: list[str]) -> tuple[pd.DataFrame, list[str]]:
    print("\n── Preprocessing ──────────────────────────────────────")
    df = pp_resample(df, raw_features)
    df = pp_iqr_clip(df, raw_features)
    df, extended = pp_rolling_statistics(df, raw_features)
    df, final    = pp_correlation_filter(df, extended)
    print(f"[preprocess] final feature count: {len(final)}")
    print("── Preprocessing complete ──────────────────────────────\n")
    return df, final

In [ ]:
# ─────────────────────────────────────────────────────────────
# SECTION 3 — SEQUENCE CONSTRUCTION
# ─────────────────────────────────────────────────────────────
def split_normal(X: np.ndarray, test_ratio: float) -> tuple[np.ndarray, np.ndarray]:
    split = int(len(X) * (1 - test_ratio))
    return X[:split], X[split:]


def make_sequences(data: np.ndarray, seq_len: int) -> np.ndarray:
    return np.array([data[i:i+seq_len] for i in range(len(data) - seq_len + 1)])


def inject_synthetic_anomalies(data: np.ndarray, ratio: float = 0.2) -> tuple[np.ndarray, np.ndarray]:
    """
    Generates OOD test samples by amplifying random feature subsets.
    Required when no labeled anomaly data exists (standard for unsupervised models).
    """
    n = int(len(data) * ratio)
    idx = np.random.choice(len(data), n, replace=False)
    anomalies = data[idx].copy()
    for a in anomalies:
        cols = np.random.choice(data.shape[-1], size=np.random.randint(2, 5), replace=False)
        a[:, cols] *= np.random.uniform(2.5, 5.0, size=len(cols))
    return (
        np.concatenate([data, anomalies]),
        np.concatenate([np.zeros(len(data)), np.ones(n)]),
    )

In [ ]:
# ─────────────────────────────────────────────────────────────
# SECTION 4 — AUTOENCODER
# ─────────────────────────────────────────────────────────────
def build_autoencoder(seq_len: int, n_features: int, latent_dim: int) -> Model:
    inp     = layers.Input(shape=(seq_len, n_features))
    x       = layers.LSTM(64, return_sequences=True)(inp)
    x       = layers.Dropout(0.2)(x)
    x       = layers.LSTM(32, return_sequences=False)(x)
    encoded = layers.Dense(latent_dim, activation="relu")(x)
    x       = layers.RepeatVector(seq_len)(encoded)
    x       = layers.LSTM(32, return_sequences=True)(x)
    x       = layers.Dropout(0.2)(x)
    x       = layers.LSTM(64, return_sequences=True)(x)
    decoded = layers.TimeDistributed(layers.Dense(n_features))(x)
    model   = Model(inp, decoded, name="lstm_autoencoder")
    model.compile(optimizer=tf.keras.optimizers.Adam(CFG["ae_lr"]), loss="mse")
    return model


def train_autoencoder(X_train: np.ndarray) -> tuple[Model, float]:
    model = build_autoencoder(X_train.shape[1], X_train.shape[2], CFG["ae_latent_dim"])
    model.summary()
    cb = [
        callbacks.EarlyStopping(patience=10, restore_best_weights=True),
        callbacks.ReduceLROnPlateau(factor=0.5, patience=5, min_lr=1e-5),
        callbacks.ModelCheckpoint(str(OUT / "ae_best.keras"), save_best_only=True),
    ]
    history = model.fit(
        X_train, X_train,
        epochs=CFG["ae_epochs"], batch_size=CFG["ae_batch"],
        validation_split=0.1, callbacks=cb, verbose=1,
    )
    recon     = model.predict(X_train, verbose=0)
    errors    = np.mean(np.mean((X_train - recon) ** 2, axis=2), axis=1)
    threshold = np.percentile(errors, CFG["anomaly_pct_threshold"])
    _plot_ae_loss(history)
    print(f"[ae] threshold (p{CFG['anomaly_pct_threshold']}): {threshold:.6f}")
    return model, threshold


def ae_reconstruction_error(model: Model, X: np.ndarray) -> np.ndarray:
    recon = model.predict(X, verbose=0)
    return np.mean(np.mean((X - recon) ** 2, axis=2), axis=1)


In [ ]:
# ─────────────────────────────────────────────────────────────
# SECTION 5 — ISOLATION FOREST
# ─────────────────────────────────────────────────────────────
def train_isolation_forest(X_flat: np.ndarray) -> IsolationForest:
    clf = IsolationForest(
        n_estimators=CFG["if_n_estimators"],
        contamination=CFG["if_contamination"],
        random_state=CFG["random_seed"],
        n_jobs=-1,
    )
    clf.fit(X_flat)
    print("[if] Isolation Forest trained.")
    return clf

In [ ]:
# ─────────────────────────────────────────────────────────────
# SECTION 6 — EVALUATION
# ─────────────────────────────────────────────────────────────
def evaluate(scores: np.ndarray, labels: np.ndarray,
             model_name: str, threshold=None) -> dict:
    auc_roc = roc_auc_score(labels, scores)
    auc_pr  = average_precision_score(labels, scores)
    precision, recall, thresholds = precision_recall_curve(labels, scores)
    f1s      = 2 * precision * recall / (precision + recall + 1e-8)
    best_idx = np.argmax(f1s)
    best_thr = thresholds[best_idx] if best_idx < len(thresholds) else thresholds[-1]
    thr      = threshold if threshold is not None else best_thr
    preds    = (scores >= thr).astype(int)
    cm       = confusion_matrix(labels, preds)

    metrics = {
        "model"    : model_name,
        "AUC-ROC"  : round(float(auc_roc), 4),
        "AUC-PR"   : round(float(auc_pr), 4),
        "Best-F1"  : round(float(f1s[best_idx]), 4),
        "Threshold": round(float(thr), 6),
    }
    print(f"\n[eval] {model_name}")
    for k, v in metrics.items():
        print(f"  {k}: {v}")

    _plot_pr_curve(precision, recall, auc_pr, model_name)
    _plot_confusion_matrix(cm, model_name)
    return metrics

In [ ]:
# ─────────────────────────────────────────────────────────────
# SECTION 7 — FEATURE IMPORTANCE
# ─────────────────────────────────────────────────────────────

def fi_autoencoder(model: Model, X_normal: np.ndarray, features: list[str]) -> pd.Series:
    """
    Per-feature mean reconstruction error on normal training data.
    High error = model struggled to reconstruct that feature = it carries
    complex signal that matters for anomaly detection.
    Shape: (N, seq_len, n_features) → mean over N and seq_len → (n_features,)
    """
    recon            = model.predict(X_normal, verbose=0)
    per_feature_err  = np.mean((X_normal - recon) ** 2, axis=(0, 1))
    return pd.Series(per_feature_err, index=features).sort_values(ascending=False)


def fi_isolation_forest(clf: IsolationForest, X_flat: np.ndarray,
                        features: list[str], seq_len: int) -> pd.Series:
    """
    Permutation importance for Isolation Forest.
    For each feature, shuffle its values across all time-step columns and
    measure the increase in mean anomaly score. Larger increase = more important.
    """
    n_features = len(features)
    baseline   = -clf.decision_function(X_flat).mean()
    importances = np.zeros(n_features)

    for fi in range(n_features):
        scores = []
        for _ in range(CFG["perm_n_repeats"]):
            X_perm   = X_flat.copy()
            feat_cols = [fi + f * n_features for f in range(seq_len)
                         if fi + f * n_features < X_flat.shape[1]]
            perm_idx  = np.random.permutation(len(X_flat))
            X_perm[:, feat_cols] = X_perm[perm_idx][:, feat_cols]
            scores.append(-clf.decision_function(X_perm).mean())
        importances[fi] = np.mean(scores) - baseline

    return pd.Series(importances, index=features).sort_values(ascending=False)


def plot_feature_importance(ae_imp: pd.Series, if_imp: pd.Series):
    """
    Side-by-side horizontal bar charts for AE and IF importance.
    Both normalised 0–1 so scales are directly comparable.
    """
    ae_norm = (ae_imp - ae_imp.min()) / (ae_imp.max() - ae_imp.min() + 1e-8)
    if_norm = (if_imp - if_imp.min()) / (if_imp.max() - if_imp.min() + 1e-8)
    if_norm = if_norm.reindex(ae_norm.index).fillna(0)   # align on AE order

    n   = len(ae_norm)
    fig, axes = plt.subplots(1, 2, figsize=(16, max(6, n * 0.38)))

    for ax, imp, cmap, title in zip(
        axes,
        [ae_norm, if_norm],
        ["RdYlGn_r", "RdYlBu_r"],
        [
            "Autoencoder\nPer-feature reconstruction error (normalised)",
            "Isolation Forest\nPermutation importance (normalised)",
        ],
    ):
        colors = plt.colormaps[cmap](np.linspace(0.15, 0.85, n))
        bars   = ax.barh(range(n), imp.values, color=colors, edgecolor="white", linewidth=0.4)
        ax.set_yticks(range(n))
        ax.set_yticklabels(imp.index, fontsize=8.5)
        ax.invert_yaxis()
        ax.set_xlabel("Importance (0–1)", fontsize=10)
        ax.set_title(title, fontsize=11, pad=10)
        ax.set_xlim(0, 1.18)
        ax.grid(axis="x", alpha=0.3)
        ax.spines[["top", "right"]].set_visible(False)
        for bar, val in zip(bars, imp.values):
            ax.text(val + 0.02, bar.get_y() + bar.get_height() / 2,
                    f"{val:.3f}", va="center", ha="left", fontsize=8)

    fig.suptitle("Feature Importance — M4 Anomaly Detection", fontsize=13, y=1.01)
    fig.tight_layout()
    fig.savefig(OUT / "feature_importance.png", dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"[importance] saved → {OUT}/feature_importance.png")

    pd.DataFrame({
        "feature"       : ae_norm.index,
        "ae_importance" : ae_norm.values,
        "if_importance" : if_norm.values,
    }).to_csv(OUT / "feature_importance.csv", index=False)
    print(f"[importance] ranked table → {OUT}/feature_importance.csv")

    print("\n[importance] Top 5 — Autoencoder:")
    for feat, val in ae_norm.head(5).items():
        print(f"  {feat:<38} {val:.4f}")
    print("\n[importance] Top 5 — Isolation Forest:")
    for feat, val in if_norm.head(5).items():
        print(f"  {feat:<38} {val:.4f}")


In [ ]:
# ─────────────────────────────────────────────────────────────
# SECTION 8 — EXPORT
# ─────────────────────────────────────────────────────────────
def export_artifacts(ae_model, ae_threshold, if_model, scaler, features, results):
    ae_model.save(OUT / "autoencoder.keras")
    joblib.dump(if_model, OUT / "isolation_forest.pkl")
    joblib.dump(scaler,   OUT / "scaler.pkl")
    meta = {
        "features"        : features,
        "sequence_len"    : CFG["sequence_len"],
        "ae_threshold"    : float(ae_threshold),
        "if_contamination": CFG["if_contamination"],
        "results"         : results,
    }
    with open(OUT / "model_meta.json", "w") as f:
        json.dump(meta, f, indent=2)
    print(f"\n[export] all artifacts → {OUT}/")
    for name in ["autoencoder.keras", "isolation_forest.pkl", "scaler.pkl", "model_meta.json"]:
        print(f"  {name}")

In [ ]:
# ─────────────────────────────────────────────────────────────
# PLOT HELPERS
# ─────────────────────────────────────────────────────────────
def _plot_ae_loss(history):
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(history.history["loss"], label="train")
    ax.plot(history.history["val_loss"], label="val")
    ax.set_title("Autoencoder Training Loss"); ax.set_xlabel("Epoch"); ax.set_ylabel("MSE")
    ax.legend(); ax.grid(alpha=0.3)
    fig.tight_layout(); fig.savefig(OUT / "ae_loss.png", dpi=150); plt.close(fig)


def _plot_pr_curve(precision, recall, auc_pr, name):
    fig, ax = plt.subplots(figsize=(6, 5))
    ax.step(recall, precision, where="post", linewidth=2)
    ax.fill_between(recall, precision, alpha=0.15, step="post")
    ax.set_xlabel("Recall"); ax.set_ylabel("Precision")
    ax.set_title(f"PR Curve — {name}\nAUC-PR={auc_pr:.4f}"); ax.grid(alpha=0.3)
    fig.tight_layout(); fig.savefig(OUT / f"pr_curve_{name.replace(' ','_')}.png", dpi=150); plt.close(fig)


def _plot_confusion_matrix(cm, name):
    fig, ax = plt.subplots(figsize=(5, 4))
    ConfusionMatrixDisplay(cm, display_labels=["Normal", "Anomaly"]).plot(
        ax=ax, colorbar=False, cmap="Blues")
    ax.set_title(f"Confusion Matrix — {name}")
    fig.tight_layout(); fig.savefig(OUT / f"cm_{name.replace(' ','_')}.png", dpi=150); plt.close(fig)


def _plot_score_distributions(ae_scores, if_scores, labels):
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    for ax, scores, title in zip(
        axes,
        [ae_scores, if_scores],
        ["Autoencoder — Reconstruction Error", "Isolation Forest — Anomaly Score"],
    ):
        ax.hist(scores[labels==0], bins=60, alpha=0.6, label="Normal",  density=True)
        ax.hist(scores[labels==1], bins=60, alpha=0.6, label="Anomaly", density=True)
        ax.set_title(title); ax.set_xlabel("Score"); ax.set_ylabel("Density")
        ax.legend(); ax.grid(alpha=0.3)
    fig.suptitle("Score Distributions — M4", fontsize=13)
    fig.tight_layout(); fig.savefig(OUT / "score_distributions.png", dpi=150); plt.close(fig)


def _plot_model_comparison(results: list[dict]):
    metrics = ["AUC-ROC", "AUC-PR", "Best-F1"]
    x, width = np.arange(len(metrics)), 0.35
    fig, ax = plt.subplots(figsize=(9, 5))
    for i, r in enumerate(results):
        bars = ax.bar(x + i * width, [r[m] for m in metrics], width, label=r["model"])
        ax.bar_label(bars, fmt="%.3f", padding=3, fontsize=9)
    ax.set_xticks(x + width / 2); ax.set_xticklabels(metrics)
    ax.set_ylim(0, 1.15); ax.set_ylabel("Score")
    ax.set_title("Model Comparison — M4 Anomaly Detection")
    ax.legend(); ax.grid(axis="y", alpha=0.3)
    fig.tight_layout(); fig.savefig(OUT / "model_comparison.png", dpi=150); plt.close(fig)



In [ ]:
# ─────────────────────────────────────────────────────────────
# MAIN
# ─────────────────────────────────────────────────────────────
def main():
    print("=" * 65)
    print("M4 — Anomaly Detection Pipeline")
    print("=" * 65)

    # 1. Load
    raw_df       = load_dataset()
    raw_features = resolve_features(raw_df)

    # 2. Preprocess
    df, features = preprocess(raw_df, raw_features)

    # 3. Scale + split
    scaler = StandardScaler()
    X_all  = scaler.fit_transform(df[features].values)
    X_train_raw, X_test_raw = split_normal(X_all, CFG["test_ratio"])

    # 4. Sequences
    seq_len  = CFG["sequence_len"]
    X_train  = make_sequences(X_train_raw, seq_len)
    X_test_n = make_sequences(X_test_raw,  seq_len)
    X_test, y_test = inject_synthetic_anomalies(X_test_n, ratio=0.2)
    print(f"[seq] train: {len(X_train):,}  |  test normal: {int((y_test==0).sum()):,}  anomaly: {int((y_test==1).sum()):,}")

    # 5. Autoencoder
    print("\n── Autoencoder ─────────────────────────────────────────")
    ae_model, ae_threshold = train_autoencoder(X_train)
    ae_scores  = ae_reconstruction_error(ae_model, X_test)
    ae_results = evaluate(ae_scores, y_test, "Autoencoder", threshold=ae_threshold)

    # 6. Isolation Forest
    print("\n── Isolation Forest ────────────────────────────────────")
    X_train_flat = X_train.reshape(len(X_train), -1)
    X_test_flat  = X_test.reshape(len(X_test),  -1)
    if_model   = train_isolation_forest(X_train_flat)
    if_scores  = -if_model.decision_function(X_test_flat)
    if_results = evaluate(if_scores, y_test, "Isolation Forest")

    # 7. Feature Importance
    print("\n── Feature Importance ──────────────────────────────────")
    print("[importance] AE — computing per-feature reconstruction error...")
    ae_imp = fi_autoencoder(ae_model, X_train, features)

    print(f"[importance] IF — permutation importance ({CFG['perm_n_repeats']} repeats)...")
    if_imp = fi_isolation_forest(if_model, X_train_flat, features, seq_len)

    plot_feature_importance(ae_imp, if_imp)

    # 8. Summary plots
    all_results = [ae_results, if_results]
    _plot_score_distributions(ae_scores, if_scores, y_test)
    _plot_model_comparison(all_results)

    # 9. Export
    export_artifacts(ae_model, ae_threshold, if_model, scaler, features, all_results)

    # 10. Final print
    print("\n" + "=" * 65)
    print("RESULTS SUMMARY")
    print("=" * 65)
    header = f"{'Model':<22} {'AUC-ROC':>9} {'AUC-PR':>9} {'Best-F1':>9}"
    print(header); print("-" * len(header))
    for r in all_results:
        print(f"{r['model']:<22} {r['AUC-ROC']:>9} {r['AUC-PR']:>9} {r['Best-F1']:>9}")
    print("=" * 65)
    print(f"\nAll outputs → {OUT.resolve()}")
    print("\nKey output files:")
    print("  feature_importance.png    ← which features drive anomaly detection")
    print("  feature_importance.csv    ← full ranked table (AE + IF side by side)")
    print("  correlation_matrix.png    ← feature correlation after filtering")
    print("  model_comparison.png      ← AE vs IF metrics bar chart")
    print("  clip_stats.json           ← IQR clipping bounds per feature")
    print("  autoencoder.keras         ← deploy to Raspberry Pi")
    print("  isolation_forest.pkl      ← deploy to Raspberry Pi")
    print("  model_meta.json           ← threshold + feature list for inference")


if __name__ == "__main__":
    main()